# UPSERTS Scenario

In [0]:
%sql
Select * from pyspark_cata_vijay.source.products

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import *
from pyspark.sql.types import *
from delta.tables import *

In [0]:
df = spark.read.table("pyspark_cata_vijay.source.products")

# DEDUP your data on the basis of UpdatedDate

df =df.withColumn('rnk',row_number().over(Window.partitionBy('id').orderBy(col('updatedDate').desc())))

df = df.filter('rnk=1').drop("rnk")

In [0]:
display(df)

**This below TRY EXCEPT method will upsert data incrementally**

In [0]:
if len(dbutils.fs.ls('/Volumes/pyspark_cata_vijay/source/pyspark_volume_practice/products/'))>0:
    dlt_obj = DeltaTable.forPath(spark,"/Volumes/pyspark_cata_vijay/source/pyspark_volume_practice/products/")

    dlt_obj.alias('trg').merge(df.alias('src'),'trg.id = src.id')\
    .whenMatchedUpdateAll(condition = "src.updatedDate >= trg.updatedDate")\
        .whenNotMatchedInsertAll().execute()
    print("this is now upserting")
else:
  df.write.format('delta').mode('overwrite').save("/Volumes/pyspark_cata_vijay/source/pyspark_volume_practice/products/")

In [0]:
%sql
Select * from delta.`/Volumes/pyspark_cata_vijay/source/pyspark_volume_practice/products/`